# 01/08 优化：梯度下降家族 — 学习 Notebook

**学习状态：进行中（2026-11-04），正式测验尚未开始。**

官方英文原文：[08-optimization/docs/en.md](../../../../../../phases/01-math-foundations/08-optimization/docs/en.md)
课程原始代码：[08-optimization/code](../../../../../../phases/01-math-foundations/08-optimization/code/)

本课沿用上一课的学习方式：学生独立作答、自己编写代码；导师逐题验收，实际执行数值测试，并保留原始答案与订正过程。

**官方课程目标**：从零实现 Gradient Descent、SGD with Momentum、Adam；使用 Rosenbrock 函数比较收敛；掌握 mini-batch、学习率调度、凸与非凸问题和鞍点。

**课程节奏**：问题与概念 → Vanilla GD → Momentum → Adam → Rosenbrock 数值实验 → PyTorch 对照 → 四项官方练习 → 三道正式后测。原始 quiz.json 中还有两道前测；正式后测前不披露正确选项。

## 第 1 讲：训练为何需要优化器？

给定参数 $w$ 和损失函数 $L(w)$，目标是：

$$\min_w L(w).$$

在可微位置，梯度 $\nabla L(w)$ 指向局部一阶增长最快的方向。最基本的**梯度下降**使用更新：

$$\boxed{w_{t+1}=w_t-\eta\nabla L(w_t)},\quad\eta>0.$$

负号保证沿负梯度方向迈步，$\eta$ 称为学习率。对于固定方向 $d=-\nabla L(w)$，有方向导数 $\nabla L(w)^\top d=-\|\nabla L(w)\|^2\le0$；但**有限步长的损失并非总会下降**，这要取决于函数局部曲率与学习率。

从本课开始，每种优化器都要区分：更新公式、使用哪些历史状态、每步计算成本、收敛速度和稳定性。用相同初始点与可比较的预算做实验，不直接把某次运行的现象概括为普遍结论。

## 第 1 讲 · 第一题：手算一次梯度下降（等待学生回答）

这是理解检查，**不是**官方测验。

给定：

$$L(w)=(w-3)^2,\quad w_0=5,\quad\eta=0.1.$$

请依次计算：

1. 初始梯度 $L'(w_0)$；
2. 更新后的参数 $w_1=w_0-\eta L'(w_0)$；
3. 更新前的损失 $L(w_0)$ 与更新后的损失 $L(w_1)$，判断是否下降；
4. 用一句话说明为什么更新公式要减去梯度，而不是加上梯度。

导师在学生回答后给出逐项验收与可运行代码，然后进入本课原始前测及 Rosenbrock 函数的梯度推导。

In [ ]:
# 01/08：代码实验从下次学生提交开始。
# 学生先手算一轮梯度下降；教师将逐项验收。


## 后续官方 Exercises 与 Post Quiz（未开始）

官方四道练习（依据英文原文，不提前填入结果）：

1. **Learning rate sweep**：Rosenbrock 上 Vanilla GD 的学习率取 0.0001、0.0005、0.001、0.005、0.01，运行 5000 步并记录最终 loss。
2. **Momentum comparison**：动量分别取 0、0.5、0.9、0.99，追踪完整 loss 曲线、收敛及振荡。
3. **Saddle point escape**：在 $f(x,y)=x^2-y^2$、初始点 $(0.01,0.01)$ 上对比 GD、Momentum、Adam。
4. **Exponential learning-rate decay**：给 GD 加入 $\eta_t=\eta_0\cdot0.999^t$，与固定学习率的 Rosenbrock 结果比较。

官方 post quiz：3 题，完成上面实验后逐题进行并登记到仓库根目录 LEARNING.md。

## 第 1 讲：第一轮梯度下降手算（学生作答与批改）

原始回答：

> 1. 梯度是4 2.w_1是4.6 3.L(w_0)=4, L(w_1) = 2.56

**数值 3/3 正确。**

设 $L(w)=(w-3)^2$，从 $w_0=5$、$\eta=0.1$ 开始：

$$L'(w)=2(w-3),\qquad L'(5)=4.$$
$$w_1=5-0.1\cdot4=4.6.$$
$$L(w_0)=4,\qquad L(w_1)=2.56.$$

损失从 4 降至 2.56。原题还要求说明**为什么沿负梯度方向而非正梯度方向**，学生本轮没有回答；该概念仍需后续理解检查，不算已验收。

第 1 讲后续：在 $L'(w_0)=4>0$ 的位置，向较小 $w$ 方向移动会使局部损失下降；一般情况下，梯度和负梯度的方向导数相反。对于较大有限学习率，并不保证损失每次都下降。

## 课程资料同步说明

学习者希望将上游项目中文译本放到个人仓库对应镜像目录，并从远端拉取至本地 WSL。

本课中文机器译本已经从上游 `translations` 分支复制到：

    learning-artifacts/cn/phases/01-math-foundations/08-optimization/docs/zh.md

**质量标记：未经逐段人工校对的中文机器译文**，上游有明显技术术语误译。本课以官方英文 docs/en.md 为准进行数学与代码校核。

学生此前已克隆 fork 到 WSL，分支是 my-progress。建议先检查本地工作区，再执行：

    cd ~/ai-engineering-from-scratch
    git status --short
    git fetch origin
    git switch my-progress
    git pull --ff-only origin my-progress

若本地 my-progress 不存在，可 `git switch --track origin/my-progress`。本次同步与前序对话中 GitHub 操作都在 fork 的 my-progress 分支，不影响 main。

继续教学时请学生补答：为什么梯度下降沿负梯度方向，而不是正梯度方向？

## 本课课程资源完整镜像与持久规则（已验收）

用户明确要求：每次进入新课，先把官方整个课程目录的**全部内容**镜像到对应中文学习目录，并把此规则写入下次能够读取的持久化项目说明，避免只复制中文讲义。

已在 `my-progress` 分支完成的文件清单：

- 原始官方文件 5/5：`docs/en.md`、`code/optimizers.py`、`code/main.jl`、`quiz.json`、`outputs/prompt-optimizer-guide.md`；均复用官方 Git blob，路径与 SHA 完全一致。
- 中文与个人文件：`docs/zh.md`（上游机器译本，质量待校对）、`quiz.zh.json`（题序、阶段和答案下标与官方一致）、`outputs/prompt-optimizer-guide.zh.md`、`code/learn.ipynb`（保留原有内容）。
- 补充入口：本课 `README.md`、`source-manifest.json`。

已写入仓库持久规则：根目录 `AGENTS.md` 指向 `learning-artifacts/cn/AGENTS.md`，该文件详细规定整课镜像、中文参考、SHA 比对、保护原始学生解答、逐题验收及完成条件；根目录 `LEARNING.md` 和中文目录 `README.md` 也有入口。

已读取 API 目录清单验证全部官方原始文件的 Git blob SHA；已核对中文版 quiz 的 5 道题在 `stage` 与 `correct` 字段上与原版完全一致。Notebook 保持原先 7 个单元后新增本条记录。

后续教学尚待学生补答：为什么梯度下降沿负梯度方向？本课尚未正式完成。

## 第 2 讲：为什么沿负梯度方向，以及 Rosenbrock 函数

第一讲学生已独立算得 $L(w)=(w-3)^2$ 在 $w_0=5$、$\eta=0.1$ 时梯度 4、更新后 $w_1=4.6$、损失由 4 降至 2.56。

一阶泰勒展开：

$$L(\boldsymbol\theta+\Delta\boldsymbol\theta)\approx L(\boldsymbol\theta)+\nabla L(\boldsymbol\theta)^\top\Delta\boldsymbol\theta.$$

对于 $\Delta\boldsymbol\theta=-\eta\nabla L(\boldsymbol\theta)$，

$$L(\boldsymbol\theta-\eta\nabla L)\approx L(\boldsymbol\theta)-\eta\|\nabla L\|_2^2.$$

负梯度方向的方向导数为负（当梯度非零）；足够小的步长可局部降低可微函数损失，但有限步长过大时不能保证下降。

本课权威英文源文件中的测试函数是：

$$f(x,y)=(1-x)^2+100(y-x^2)^2.$$

全局最小值为 $f(1,1)=0$，其附近的狭窄曲线谷地会给优化器带来迂回和振荡问题。

### 学生下一题（待回答）

独立推导 $\partial f/\partial x$ 和 $\partial f/\partial y$，说明对 x 求偏导时如何使用链式法则。**不要提前填写标准导数答案**。学生提交后与解析梯度及有限差分测试交叉验证，之后再让学生自己实现 `rosenbrock_gradient(params)`。

本课知识和代码以后均对照同目录的原始 `docs/en.md`、`code/optimizers.py` 和官方中文学习材料；继续遵循 `learning-artifacts/cn/AGENTS.md` 中完整镜像与逐题验收规则。

## 01/08 · Rosenbrock 偏导验收

学生独立提交：

\[
\frac{\partial f}{\partial x}=-2(1-x)+200(y-x^2)(-2x),\qquad
\frac{\partial f}{\partial y}=200(y-x^2).
\]

**判定：两个偏导均正确。** 化简后：

\[
\nabla f(x,y)=
\begin{pmatrix}
-2(1-x)-400x(y-x^2)\\
200(y-x^2)
\end{pmatrix}.
\]

已在独立 Python 环境实际运行中心差分检查，6 个测试点乘以 2 个偏导共 **12/12 通过**，点为 (-1,1)、(1,1)、(0,0)、(1.4,2.2)、(-0.3,0.8)、(2,-3)，差分步长 1e-6。下个代码单元提供复现用的检查，未把教师程序冒充学生代码。

## 本课中文教材精译说明

已完整重译同课文档 zh.md，保留与英文版一致的 24 个标题、17 个代码及图示块和 4 个外部参考网址。对原文容易造成误解的优化理论陈述以译注注明适用条件。旧的低质量机器译本已替换，英文原文与个人学习记录未覆盖。

**下一步待学生完成：** 自主将偏导表达式写成 Python 函数 rosenbrock_gradient(params)，返回两个偏导组成的列表；教师随后进行实际运行与数值差分验收。

In [ ]:
import math
def f(x, y):
    return (1-x)**2 + 100*(y-x*x)**2
def grad(x, y):
    return (-2*(1-x)+200*(y-x*x)*(-2*x), 200*(y-x*x))
points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in points:
    gx,gy = grad(x,y)
    dx = (f(x+h,y)-f(x-h,y))/(2*h)
    dy = (f(x,y+h)-f(x,y-h))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7)
    checks += 2
print(f'Central-difference gradient checks: {checks}/12 passed')

## 学生独立实现：rosenbrock_gradient(params)（通过）

学生原始代码见下一单元。该实现把此前的解析偏导正确转换为 Python，返回 `[df_dx, df_dy]`，`params` 的两个坐标解包顺序也正确。

在 Python 中已实际执行 14/14 项测试：六个点各两项中心有限差分共 12 项，另检验两个解析点 `grad([1,1]) == [0,0]` 及 `grad([-1,1]) == [-4,0]`。中心差分步长 `1e-6`；判定使用 `math.isclose(rel_tol=3e-8, abs_tol=4e-7)`。

**结论：该函数通过本轮验收。** 不需修改学生代码。

## docs/zh.md 格式修复

检查出中文精译版曾使用 `\[` / `\]` 块级公式定界符、五个无语言标签的代码围栏，以及一处仅供课程网站使用的 `figure` 标识。

已改为 GitHub 推荐的独立 `$$` 数学块、增加 `text` 代码标签，且把 `figure` 专用标签换成可阅读的文本与兼容说明。原文 17 个围栏块的**主体内容**保持一致，24 个标题保持一致，英文源文件不变。

**下一步**：根据官方课程 Build It / Step 2，自主实现 `GradientDescent.step(params, grads)`；确认数学方向和输入输出后再引入动量。

In [ ]:
def rosenbrock_gradient(params):
    # params = [x, y]
    # 返回 [df_dx, df_dy]
    x,y = params
    df_dx =  -2 * (1 - x) - 400 * x * (y - x ** 2)
    df_dy = 200 * ( y - x ** 2)

    return [df_dx, df_dy]

In [ ]:
import math

def rosenbrock_value(point):
    x, y = point
    return (1-x)**2 + 100*(y-x*x)**2

check_points = [(-1,1),(1,1),(0,0),(1.4,2.2),(-0.3,0.8),(2,-3)]
h = 1e-6
checks = 0
for x,y in check_points:
    gx, gy = rosenbrock_gradient([x,y])
    dx = (rosenbrock_value((x+h,y))-rosenbrock_value((x-h,y)))/(2*h)
    dy = (rosenbrock_value((x,y+h))-rosenbrock_value((x,y-h)))/(2*h)
    assert math.isclose(gx,dx,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dx',gx,dx)
    assert math.isclose(gy,dy,rel_tol=3e-8,abs_tol=4e-7), (x,y,'dy',gy,dy)
    checks += 2
assert rosenbrock_gradient([1,1]) == [0,0]
assert rosenbrock_gradient([-1,1]) == [-4,0]
checks += 2
assert checks == 14
print('Rosenbrock gradient: 14/14 PASS')

## 下一项独立编码练习：GradientDescent（等待提交）

公式：`new_p = p - lr*g`（逐坐标计算）。

请只写如下类的 `__init__` 与 `step`，不要先看官方答案：

    class GradientDescent:
        def __init__(self, lr=0.001):
            pass

        def step(self, params, grads):
            pass

`params` 和 `grads` 是长度相同的序列，`step` 返回更新后的参数列表，不需要计算梯度。提交后通过多个维度、正负梯度和不同学习率进行测试。

In [ ]:
class GradientDescent:
        def __init__(self, lr=0.001):
            self.lr = lr

        def step(self, params, grads):
            new_p = params - self.lr * grads
            return new_p

## GradientDescent：学生首次代码提交与测试结果（需订正）

学生独立代码见上方单元；已完整保留原始写法。

**验收：`__init__` 正确；`step` 在要求的 Python 列表输入下失败，暂未通过。**

数学更新公式正确，但代码 `new_p = params - self.lr * grads` 是对 `list` 应用数组向量化运算。

导师实际运行输入 `params=[5.0,3.0]`, `grads=[4.0,-2.0]`, `lr=0.1`：

```text
TypeError: can't multiply sequence by non-int of type 'float'
```

同样的代码若输入为 `numpy.ndarray`，独立实际运行输出 `[4.6 3.2]`，与数学预期一致；但这不是本课程要求的 list 输入。

**待学生独立修复：** 使用 `zip(params, grads)` 逐坐标读取 `p,g`，计算 `p-self.lr*g` 并返回新列表；不提前提供最终完整类答案。接下来检验二维与多维输入、正负梯度、不同学习率。

该错误和订正要求必须保留，不因最后修正成功而删除原始版本。

In [ ]:
class GradientDescent:
        def __init__(self, lr=0.001):
            self.lr = lr

        def step(self, params, grads):
            new_p = [param - self.lr * grad for param, grad in zip(params, grads)]
            return new_p

## GradientDescent 订正完成：通过

学生修订版代码保存在上方独立代码单元。修正了第一次使用 Python list 直接进行向量乘减的错误，改为利用 `zip` 配对参数与梯度，以列表推导式逐坐标更新。

**判定：正确，通过官方 Build It 第 2 步。** 计算公式：

$$\theta_{i,t+1}=\theta_{i,t}-\eta g_{i,t}.$$

本轮实际在 Python 中执行 **6/6 组测试通过**：二维正负混合梯度、默认学习率的三维更新、一维更新、零梯度、多维负参数、tuple 输入返回列表。使用 `math.isclose` 比较浮点数，容差均为 1e-12。

备注：Python `zip` 默认按较短序列截断；本课给定参数与梯度等长的输入契约。生产实现可以增加长度检验，以免悄悄丢失参数。

此前失败的首次提交仍保留在前序单元，未覆盖。

In [ ]:
import math

gd_cases = [
    ('2d-mixed', 0.1, [5.0,3.0], [4.0,-2.0], [4.6,3.2]),
    ('default-3d', 0.001, [1.0,2.0,3.0], [2.0,0.0,-2.0], [0.998,2.0,3.002]),
    ('1d', 0.5, [2.0], [-3.0], [3.5]),
    ('zero-gradient', 0.3, [0.0,10.0,-3.0], [0.0,0.0,0.0], [0.0,10.0,-3.0]),
    ('negative-params', 0.2, [-10.0,5.0,0.5], [5.0,1.0,-0.5], [-11.0,4.8,0.6]),
    ('tuple', 0.1, (5.0,3.0), (4.0,-2.0), [4.6,3.2]),
]
for label, lr, p, g, expected in gd_cases:
    got = GradientDescent(lr).step(p, g)
    assert isinstance(got,list) and len(got)==len(expected), label
    assert all(math.isclose(a,b,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(got,expected)), label
print('GradientDescent correction: 6/6 PASS')

## 官方 Build It 第 3 步：Momentum（待学生）

本课官方的状态更新为：

$$v_t=\beta v_{t-1}+g_t,\qquad\theta_{t+1}=\theta_t-\eta v_t.$$

在第一次调用 `step` 时，把 `velocity` 初始化为与 `params` 同维度的全零列表；随后每次 `step` 都更新并保留 `velocity`。`beta` 即动量系数。

**下一项理解检查：** 假设一维参数 $w_0=5$，$v_0=0$，$\eta=0.1$，$\beta=0.9$，连续两步收到相同梯度 $g_1=g_2=4$。手算第一步、第二步的 $(v,w)$，并指出它与两步普通梯度下降的差别。学生回答后再进入 Momentum 类的代码实现。

保持逐题推进，不提前向学生给出这道手算题的数值答案。

## Momentum 手算验收

学生原始答案：

> v_1 =  4, w_1 = 4.6, v_2 = 7.6,w_2 = 3.84

**4/4 正确。**

按本课动量更新公式：

$$v_t=\beta v_{t-1}+g_t,\quad w_t=w_{t-1}-\eta v_t,$$

代入 $w_0=5$, $v_0=0$, $\beta=0.9$, $\eta=0.1$, $g_1=g_2=4$：

$$v_1=4,\quad w_1=4.6,\quad v_2=0.9\cdot4+4=7.6,\quad w_2=4.6-0.76=3.84.$$

普通 GD 两步后得到 $w_2=4.2$；动量法第二步使用历史梯度，单步更新幅度 0.76，大于普通 GD 的 0.4。

这是额外的理解检查，非官方 Exercises 或 Quiz。

## 官方流程审计：补全此前遗漏（2026-11-06）

学生要求严格遵循课程。现核对本课官方 docs/en.md、quiz.json 及仓库 AGENTS.md 与个人学习规约。

**目前不算完全严格：** 原始 5 个文件已完整镜像；Build It 第 1 步 Rosenbrock 和第 2 步 GradientDescent 已完成并验收；Build It 第 3 步 Momentum 只完成手算，还未完成类实现。

**明确遗漏：** 原版 quiz.json 的两道 stage=pre 在教学开始前没有出题。现在只能按“迟做的前测题”补做，不能当成真实课前知识基线。

**尚未完成：** Build It 第 4 步 Adam、第 5 步完整对比；Use It、Ship It；四道官方 Exercises；三道 stage=post 正式后测。

此前手算、数值差分和附加测试是有价值的教学补充，但不能替代官方练习。整课仍未完成，不更新 LEARNING.md 完成状态。

接下来先补做原始两道前测（按官方题号，每次一题，清楚标明补做），然后恢复 Momentum 类实现。

## 官方前测第 1/2 题（迟做，等待学生回答）

本题原来自 quiz.json 中的第 1 道 stage=pre，因此前教学已开始，这次只作补题，不作为真正的学习前诊断成绩。

**训练神经网络时，“优化”（optimization）具体意味着什么？**

A. 寻找使损失函数最小的模型参数

B. 使程序代码运行得更快

C. 减少模型参数的数量

D. 选择更好的训练硬件

请学习者先给出选项与理由。评分后再呈现第二道前测，未答之前不写出正确选项。

## 01/08 · 原本的前测第 1 题补答记录

学生原始回复：

> 01/08 官方补做前测第1题：选择 A。理由：不对,你还漏了讲解的部分,你要先读取它的prompt,然后再慢慢讲解

选项 **A 正确（1/1）**：训练中的优化是寻找使损失函数尽可能小的模型参数。学生的后半段是指出教学流程错误，并不是对选项的概念解释，不应伪称已经给出数学理由。

按 Skill 审计：`skills/learn/SKILL.md` 在 Step 2 要求先讲 Problem/Concept、先暂停理解题，再逐步推导和分段讲代码；Step 3 正式进行的是 `stage=post` 的题目。原版 `quiz.json` 中的 `pre` 题不是该 Skill 的强制门槛。之前“必须完成全部 pre 才能讲下一节”的说法错误，已在中文 AGENTS.md 修正。前测第 2 题不强制补做。

## 本课 Prompt 的实际用途与恢复教学计划

- **教学 Skill**：`skills/learn/SKILL.md`（仓库原始教学规范），适用于所有课程；必须先读取并遵循交互式教学流程。
- **本课英文讲义**：`docs/en.md`（课程数学与代码权威）；其中文镜像为 `docs/zh.md`。
- **Ship It Prompt**：`outputs/prompt-optimizer-guide.md`（为具体 ML 任务选择优化器、学习率和 schedule 的交付提示词），并非授课提示词；到 Ship It 阶段必须讲解与实际使用它。

在本章第 1、2 步 Build It 已完成、Momentum 手算正确的情况下，用户指出解释不足。因此先回到漏讲的 Concept 部分，把**梯度和优化器的区别、learning rate、batch/SGD/mini-batch、为什么动量能减轻狭窄谷地振荡**分小节互动讲透，然后继续 Momentum 类的独立实现。教学过程每次只讲一小节，不一次性塞进所有公式与代码。

这条安排用于纠正教学节奏，不清空任何已提交代码，不将新增理解练习算作官方考试。

## 恢复 `learn` 课程：核心概念理解检查（等待学生）

### 为什么已经知道梯度，还需要优化器？

把梯度理解为当前位置的局部坡度，优化器是决定**如何利用坡度、历史方向、尺度**来更新模型参数的规则。普通 GD 只看当前梯度，Momentum 还会利用过去若干步的历史信息。

在 Rosenbrock 狭窄弯曲的谷地中，**横穿谷地**的梯度方向可能频繁反转，**沿谷地前进**的分量在若干步内却可能保持一致。因此普通 GD 容易左右来回跳，Momentum 会使反转方向的历史项相互抵消、在一致方向上积累速度。这解释了为什么不同方向的优化速度可能不同。

**请先用文字回答：** 如果梯度连续两步在某个方向上正负交替，另一个方向上一直保持同号，Momentum 对这两个方向分别可能产生什么影响？

学生回答后再推导对应分量的更新，接着解释 batch vs mini-batch；此刻不要催促直接写完整 SGDMomentum 类。

## Momentum：历史梯度与各坐标振荡的理解检查（已批改）

学生原始答案：

> momentum会导致 梯度在x方向减左右震荡的频率,会增加y方向的梯度

**方向判断基本正确，但两处表述需订正：**

1. x 方向：动量加权累积可以抑制正负交替梯度带来的更新**幅度**，不能保证改变交替的**频率**。
2. y 方向：动量不会直接增大当前真实梯度 $g_t$；它通过累积相同方向的历史梯度增大速度 $v_t$，从而可能放大参数更新。

本课动量采用 $v_t=\beta v_{t-1}+g_t$，$\theta_t=\theta_{t-1}-\eta v_t$，初始 $v_0=(0,0)$。设 $\beta=0.9$，梯度 $g_1=(4,1)$，$g_2=(-4,1)$，则：

$$v_1=(4,1),\qquad v_2=0.9(4,1)+(-4,1)=(-0.4,1.9).$$

因此第二步原始梯度是 $(-4,1)$，但有效更新向量是 $-\eta(-0.4,1.9)$。展开递推：

$$v_t=\sum_{k=0}^{t-1}\beta^k g_{t-k}\quad(v_0=0).$$

该解释对一致方向的积累与振荡方向的部分抵消给出数学依据，但并非对任意梯度序列均保证抑制振荡或改善收敛。测试单元复算两步速度。

**下一步补充理解题**：从 $v_2=(-0.4,1.9)$ 出发，如果下一步 $g_{3,y}=-1$，且 $\beta=0.9$，预测第 3 步的 $v_{3,y}$ 是正、负还是零？解释为何梯度已经反向，参数更新方向却可能尚未反向。学生答完再进入 Momentum 类实现。

In [ ]:
beta = 0.9
velocity = [0.0, 0.0]
observed = []
for gradients in ([4.0, 1.0], [-4.0, 1.0]):
    velocity = [beta * v + g for v, g in zip(velocity, gradients)]
    observed.append(velocity[:])
assert all(abs(a-b)<1e-12 for got,expected in zip(observed,[[4.0,1.0],[-0.4,1.9]]) for a,b in zip(got,expected))
print('Momentum two-step concept test: PASS', observed)

## Momentum：当前梯度反向而速度保持同向（通过）

学生原始回答：

> v3,y是正数,因为前面速度加新的梯度没有反向

**判定：正确。** 学生理解历史动量会阻止更新方向立即跟随瞬时梯度反转。

根据上一个理解检查 $v_{2,y}=1.9$，$g_{3,y}=-1$，$\beta=0.9$：

$$v_{3,y}=0.9\times1.9-1=0.71>0.$$

若学习率 $\eta>0$，则参数更新 $\Delta y_3=-\eta v_{3,y}=-0.71\eta<0$，即**速度仍为正、参数仍向较小的 y 移动**。当前梯度负号不等于瞬时速度立刻反向；注意梯度与速度是不同变量。

这一题是 Concept 的补充理解检查，不计入官方 Post Quiz 或 Exercises。

## 官方 Build It 第 3 步：Momentum 类——先完成状态初始化

解释：普通 `GradientDescent.step()` 仅保存学习率，Momentum 还需要在**同一个对象**里持续保留历史速度，因此不能每次 `step` 都重新初始化速度。

官方初始化接口：

```python
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        # 保存学习率、动量系数和速度状态
        ...
```

学生本轮只需要独立完成 `__init__`，先不写 `step` 的最终实现。要求将 `lr`、`momentum` 保存为对象属性；`velocity` 初始为 `None`，代表还不知道参数维度、尚未初始化。之后再讨论首次 `step` 如何构造长度相同的零向量和继续更新的区别。

理解问题：**为什么速度不能在每一次 `step()` 开头都设为零？** 学生写好初始化方法并解释后再进入更新逻辑。

后续仍需要完成未充分讲解的 SGD / batch / mini-batch 的 Concept，以及 Adam、LR schedule、凸/非凸、损失地形，再做完整对照实验和官方练习。按官方 `skills/learn/SKILL.md` 概念优先、5～15 行分段互动，不跳过。

In [ ]:
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        # 由你完成
        self.lr = lr
        self.momentum = momentum
        self.velocity = 0

## SGDMomentum 第一版初始化：需要订正

学生原始代码保存在上一单元，聊天附带解释为：

> 因为我们的速度要用前面的速度更新,不能设置为0

**评价**：`lr`、`momentum` 两个属性初始化正确；理解到速度必须保留历史状态。`velocity=0` 是标量整数，和官方类的初始状态约定不一致。

本课官方 `code/optimizers.py` 中的 `SGDMomentum.__init__` 使用 `self.velocity = None`；第一次进入 `step` 时按参数维度初始化零列表，之后复用速度。零向量只应在初次使用时设置，而不是每一步重置。将它初始化为整数 0 会导致 `if self.velocity is None` 无法触发，且后续在 `zip(self.velocity, grads)` 中因整数不可迭代而失败。

**实际 Python 检查结果**：构造 `SGDMomentum()` 得到 `lr=0.001`、`momentum=0.9`、`velocity=0`；`velocity is None` 为 False；以 0 和梯度列表运行 `zip` 抛出 `TypeError: 'int' object is not iterable`。

**订正要求**：只将初始化阶段的速度改成 `None`，说明 `None` 是尚未建立速度列表的状态标识，以及为什么不能在 `__init__` 中根据未知参数维度预先创建适当长度的零列表。先不要求学生完成整个 `step`。

学生在消息中与代码同一行书写的中文是回答解释，不作为可执行的 Python 源码，Notebook 将其保留在此。

## Momentum 延迟初始化理解检查

学生原始回答：因为我们不知道我们要调整的速度的列表的形状,我们后面根据参数的维度创建零向量

**理解正确。** `None` 表示尚未初始化，首次 `step` 根据 `len(params)` 建立同维零向量。此前代码 `self.velocity=0` 尚未由学生订正。

下一步让学生独立完成 `step` 中检测 `self.velocity is None` 并建立零向量的条件分支，之后才写动量递推。

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        self.velocity = [0.0] * len(params)

## Momentum step 首次初始化分支——第一次提交（需订正）

学生原始代码保存在上一个代码单元。

**判断：** `[0.0] * len(params)` 构造同维零向量正确，但没有判断 `self.velocity is None`，因此每次调用 `step` 都清除历史速度，破坏 Momentum 的跨步状态。

订正要求：仅在速度尚未初始化（`self.velocity is None`）时创建零列表；否则保留历史速度。先不实现速度递推和参数更新。学生独立订正后再验收。

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        if self.velocity == None:
            self.velocity = [0.0] * len(params)

## SGDMomentum `step()` 初始化条件：第二次提交通过

学生原始版本见上一代码单元（本次使用 `== None`，而非官方的 `is None`）。

**本轮局部任务通过。** `if self.velocity == None` 在当前对象状态为 `None` 或普通 Python 列表时可以实现仅初始化一次的效果，之后不会清零。对 `None` 的身份测试推荐写作 `is None`，语义更清楚，且避免数组类型的逐元素比较歧义。

实际 Python 执行 2/2 通过：

1. `velocity=None`，`params=[1.,2.,3.]`，调用后获得 `[0.,0.,0.]`；
2. 人为将 `velocity=[4.,2.,-1.]`，再次调用仍保持该值，不被清零。

注意：**这仅说明初始化分支完成；整个 `step` 尚未完成**，不应把完整的 Momentum 优化器标记为通过。

下一步只实现公式 `v_t=beta*v_{t-1}+g_t` 的逐参数列表更新，复用上节学到的 `zip`。速度更新后再独立实现参数更新。

In [ ]:
class _MomentumInitOnly:
    def __init__(self):
        self.velocity = None
    def step(self, params, grads):
        if self.velocity == None:
            self.velocity = [0.0] * len(params)

instance = _MomentumInitOnly()
instance.step([1.,2.,3.], [1.,1.,1.])
assert instance.velocity == [0.,0.,0.]
instance.velocity = [4.,2.,-1.]
instance.step([1.,2.,3.], [1.,1.,1.])
assert instance.velocity == [4.,2.,-1.]
print("Momentum state initialization: 2/2 PASS")

In [ ]:
def step(self, params, grads):
        # 请你完成首次初始化判断
        if self.velocity == None:
            self.velocity = [0.0] * len(params)
        self.velocity = [self.momentum * v + g for v,g in zip(self.velocity,grads)]
        new_p = [param - self.lr * v for param, v in zip(params, self.velocity)]

        return new_p

## SGDMomentum 完整 step：验收通过，注意构造器前置条件

学生提交的 step 原始内容保存在上一单元，已经正确实现一次性状态初始化、速度递推及参数更新。
实际在 Python 环境运行 13/13 检查通过，涉及两步 1D、三步 2D、正负梯度反转、零动量、tuple 和输入对象不变性。下一个代码单元可复现。

**重要：本次只提交 step 函数。测试假设构造器为之前演示的正确形式 self.velocity = None；此前学生提交的构造器还曾是 self.velocity = 0，尚未提交新的完整构造器。若仍用 0，则旧状态判定不会触发，随后 zip(self.velocity,grads) 抛出 TypeError。** 因此正式判断是 step 已通过，整个类还需确认将构造器改为 None。

建议将 if self.velocity == None 改成 if self.velocity is None（None 单例的标准身份判断）。官方 Rosenbrock 实验的 gradients 是完整解析梯度，不包含随机采样，此处需要与 mini-batch SGD 区分。

In [ ]:
import math
class SGDMomentum:
    def __init__(self, lr=0.001, momentum=0.9):
        self.lr, self.momentum, self.velocity = lr, momentum, None

    def step(self, params, grads):
        if self.velocity == None:
            self.velocity = [0.0] * len(params)
        self.velocity = [self.momentum * v + g for v,g in zip(self.velocity,grads)]
        new_p = [param - self.lr * v for param, v in zip(params, self.velocity)]
        return new_p

results=[]
def check(name,got,expected):
    assert len(got)==len(expected)
    assert all(math.isclose(a,b,rel_tol=1e-12,abs_tol=1e-12) for a,b in zip(got,expected)),(name,got,expected)
    results.append(name)
m=SGDMomentum(.1,.9)
w1=m.step([5.],[4.]); check('1d-first-w',w1,[4.6]); check('1d-first-v',m.velocity,[4.])
w2=m.step(w1,[4.]); check('1d-second-w',w2,[3.84]); check('1d-second-v',m.velocity,[7.6])
m=SGDMomentum(.1,.9)
w1=m.step([1.,2.],[4.,1.]); check('2d-first-w',w1,[.6,1.9]); check('2d-first-v',m.velocity,[4.,1.])
w2=m.step(w1,[-4.,1.]); check('2d-second-w',w2,[.64,1.71]); check('2d-second-v',m.velocity,[-.4,1.9])
w3=m.step(w2,[0.,-1.]); check('2d-third-w',w3,[.676,1.639]); check('2d-third-v',m.velocity,[-.36,.71])
m=SGDMomentum(.2,0); check('zero-momentum',m.step([3.,4.],[-2.,3.]),[3.4,3.4])
m=SGDMomentum(.1,.9); p=[1.,2.]; g=[.2,-.5]; m.step(p,g); assert p==[1.,2.] and g==[.2,-.5]; results.append('non-mutating')
m=SGDMomentum(.1,.9); check('tuple-input',m.step((5.,3.),(4.,-2.)),[4.6,3.2])
assert len(results)==13
print('SGDMomentum: 13/13 PASS')

## 概念理解检查：SGD 与 mini-batch

官方类名是 SGDMomentum，但在 Rosenbrock 实验中提供给 step 的是完整解析梯度；优化器如何更新参数，与调用者如何估计梯度，是两个层面。

问题：如果有 1000 个训练样本，每一步随机取 32 个计算梯度并调用你的 SGDMomentum.step(params,grads)，这属于全批量、单样本 SGD 还是 mini-batch SGD？相比使用全部 1000 个样本，梯度具有什么特点？

按一问一答等待学生回复，随后进入官方 Adam 概念与数学推导。

## 概念理解检查：SGD / Mini-batch（通过）

学生原始回答：

> 这个属于mini-batch sgd 相对于1000个样本点全批量梯度下降这个梯度的特点就是快些,然后噪声是中等,但是我要顺便问一下sgd的s是什么英文

**判定：正确。** 每次随机抽取32/1000样本计算梯度属于 Mini-batch SGD；相较全批量，单次梯度计算通常成本更低，并存在样本抽样噪声。需注意总收敛时间未必更短，噪声大小取决于批大小和样本梯度分布，不能固定称为“中等”。

学生询问的术语：`SGD` = **Stochastic Gradient Descent**；`S` = **Stochastic**，意为随机的。

若完整训练集目标为 $L(\theta)=\frac{1}{1000}\sum_{i=1}^{1000}\ell_i(\theta)$，其梯度为

$$g_{\mathrm{full}}=\frac{1}{1000}\sum_{i=1}^{1000}\nabla\ell_i(\theta).$$

每次均匀抽取大小为32的 batch $B$，

$$g_B=\frac{1}{32}\sum_{i\in B}\nabla\ell_i(\theta),\qquad\mathbb E_B[g_B]=g_{\mathrm{full}}.$$

这里的无偏等式要求均匀抽样及恰当的样本均值估计；不表示单次小批量梯度等于完整梯度。该部分是 Concept 课堂理解检查，不计入官方练习/后测。

### 下一步（等待学生回答）

进入 Adam 的概念：相同学习率下，不同参数的梯度历史尺度可能不同。向学生提出直观问题：如果两个参数的梯度长期分别约为 100 和 0.01，而使用同一个学习率，更新尺度会有什么差异，可能带来什么优化问题？先回答直觉，再推导 Adam 的一阶、二阶矩及偏差修正。

## Adam 自适应学习率动机：更新幅度计算（通过）

学生原始回答：

> 参数a的更新幅度是0.1 参数B的更新幅度是0.00001

**数值计算 2/2 正确。** 固定学习率 $\eta=0.001$，梯度 $g_A=100$、$g_B=0.01$：

$$\Delta\theta_A=-0.001(100)=-0.1,\quad \Delta\theta_B=-0.001(0.01)=-10^{-5}.$$

更新幅度相差 $10^4$ 倍。在相同学习率下，参数的梯度尺度差异会导致参数更新尺度差异，但不能仅凭尺度不同便认定有问题，还要结合曲率与参数尺度。

**该题概念解释部分待答，不能标记整题全部完成。** 下一问：若增大学习率使 B 移动更快，A 有什么潜在风险？若缩小学习率确保 A 稳定，B 又会怎么样？等学生回答后再深入讲解 Adam 的一阶矩、二阶矩和偏差修正。

## Adam 动机：固定学习率两难（理解检查通过）

学生原始回答：

> 如果B更新的更快,那么A的更新幅度也跟着变大,可能会跳过最优点,左右跳动,如果A学习率调小,B就变得更小了,可能需要很多很多step才能缓慢移动

**结论：正确。** 使用相同学习率、梯度分别为 $100$ 与 $0.01$ 时，放大全局学习率会同步放大两个参数的更新，可能令大梯度方向越过低损失区并振荡甚至发散；缩小学习率可能导致小梯度方向移动缓慢。该现象是否发生取决于方向曲率、参数尺度与具体目标，不是必然发生。

学生已完成公式计算 $|\Delta\theta_A|=0.1$、$|\Delta\theta_B|=10^{-5}$ 和含义解释两个部分；本轮理解检查通过。

### Adam 结构初步讲解与下一道理解检查

官方课程提出：Adam 为每个坐标分别维护两个历史统计量：

1. **一阶矩估计** $m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$：带符号的梯度滑动平均，反映历史上的更新方向。
2. **二阶原始矩估计** $v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$：梯度平方的滑动平均，反映历史梯度量级（不是方差）。

它们与 SGDMomentum 课程之前的无归一化速度 $u_t=\beta u_{t-1}+g_t$ 不应混同：Adam 的一阶矩使用 $(1-\beta_1)$ 权重，且随后有偏差修正。

下一问：若某参数的梯度交替为 $+10,-10,+10,-10,\ldots$，为什么直接对梯度取滑动平均可能互相抵消，而对梯度平方取滑动平均不会？只解释直觉；之后再做一阶矩、二阶矩和偏差修正的数值推导。

## Adam：一阶矩、二阶原始矩的含义和区别

学生原始回答及提问：

> 按照我的理解一阶矩,它的梯度是g_t是可以看到正负的,代表着方向,二阶矩的g_t ^ 2 只有大小,代表着梯度的量级,我需要你给我讲一下二阶矩一节矩,

**理解判断：正确。** 一阶矩保留梯度符号，反映梯度的平均方向；二阶原点矩通过平方反映梯度的平方尺度，不受正负号抵消影响。注意它不是平均绝对值，也不是方差。对随机变量 X，

$$\mu_1'=\mathbb E[X],\quad\mu_2'=\mathbb E[X^2],\quad\operatorname{Var}(X)=\mathbb E[X^2]-(\mathbb E[X])^2.$$

例：四个梯度 [10,-10,10,-10]，平均梯度 0，但平均平方梯度 100，方差 100；四个梯度 [10,10,10,10]，平均 10、平均平方 100、方差 0。二阶原点矩无法单独反映梯度波动。

Adam 的逐坐标指数移动平均（设对应坐标的第 t 步梯度为 g_t）：

$$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t,$$

$$v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2.$$

其中 m 是一阶矩估计，v 是二阶原始矩估计；它们是对不断变化的训练梯度的历史加权估计，并非任何时刻都严格等于固定分布下的真期望。常用 \beta_1=0.9, \beta_2=0.999。

初始化 m_0=v_0=0 时，早期 EMA 存在趋近零的初始化偏差。Adam 使用

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad\hat v_t=\frac{v_t}{1-\beta_2^t},$$

$$\theta_t=\theta_{t-1}-\eta\frac{\hat m_t}{\sqrt{\hat v_t}+\epsilon}.$$

这里 \sqrt{\hat v_t} 是历史梯度均方根尺度的估计；这一项帮助根据每个坐标的梯度历史尺度归一化更新。偏差修正在梯度同分布且期望固定的理想化条件下可推导期望的初始缩小因子；训练中梯度随参数变化，因而不是对所有情形的精确无偏保证。

**单步示例：** 如果 m_0=v_0=0、g_1=10，则 m_1=1，v_1=0.1，\hat m_1=10，\hat v_1=100。

**下一次交互问题（等待学生回答）：** 为什么第一步 m_1=1、v_1=0.1，而偏差修正后分别成为 10 和 100？请用 EMA 的初始值解释。答完后进入第二步正负交替梯度示例与 Adam 代码分段实现。

## Adam EMA 与偏差修正：概念讲解（2026-11-06）

### 学生原始回答与提问

> 因为是从零初始化的,早期估计偏向零,我需要你给我讲一下为什么要这样修正,什么是估计,为什么叫指数移动平均

**理解检查：通过。** 学生正确理解零初始化引起早期估计趋零，并主动要求解释估计、指数移动平均及修正的来源。

### 1. 统计估计与 Adam 的状态

对于固定分布下随机变量 $X$，未知一阶原点矩 $\mu=\mathbb E[X]$ 可由样本平均 $\hat\mu_n=\frac{1}{n}\sum_i X_i$ 估计；二阶原点矩 $\mathbb E[X^2]$ 可由平方样本平均估计。估计量是样本到数值的函数，估计值是代入具体样本后的结果。

Adam 对每个参数的历史梯度保留两种时变统计量：$m_t$ 跟踪带符号梯度的指数平均，$v_t$ 跟踪平方梯度的指数平均；在非平稳训练过程中，它们不是对一个固定真实期望的保证无偏估计。

### 2. 为什么是指数移动平均

$$m_t=\beta m_{t-1}+(1-\beta)g_t,\quad m_0=0.$

$$m_t=(1-\beta)\sum_{k=1}^{t}\beta^{t-k}g_k.$$

最近的 $g_t$ 权重 $(1-\beta)$，倒数第二个为 $(1-\beta)\beta$，倒数第三个为 $(1-\beta)\beta^2$，依次指数衰减。它随每个新观察值持续更新，因此称为“移动”平均，无需存储全部梯度。

### 3. 从权重之和严格推导偏差修正

$$\sum_{k=1}^t(1-\beta)\beta^{t-k}=(1-\beta)\sum_{j=0}^{t-1}\beta^j=1-\beta^t.$$

由于 $m_0=0$，在早期权重总和小于 1；对恒定梯度 $g_k=c$，有 $m_t=c(1-\beta^t)$，并不是 $c$。因此归一化：

$$\hat m_t=\frac{m_t}{1-\beta_1^t},\qquad \hat v_t=\frac{v_t}{1-\beta_2^t}.$$

更一般地，在各步梯度满足同一个期望 $\mathbb E[g_k]=\mu$ 的条件下，有 $\mathbb E[m_t]=(1-\beta^t)\mu$，故修正后 $\mathbb E[\hat m_t]=\mu$；即便样本非独立，这一线性期望结论仍成立。若梯度期望随训练发生变化，归一化只修正零初始化产生的缩小因子，并不保证对当前梯度期望完全无偏。

### 4. 数字例子

取 $\beta=0.9$，$m_0=0$，每次 $g_k=10$。递推得到 $m_1=1$、$m_2=1.9$、$m_3=2.71$、$m_4=3.439$；除以 $1-0.9^t$ 后四次均为 10。

二阶矩取 $\beta_2=0.999$，若 $g_1=10$，则 $v_1=0.001\times100=0.1$，$\hat v_1=0.1/(1-0.999)=100$。二阶矩不等于方差，其平方根对应梯度均方根尺度。

### 下一道理解检查（等待学习者）

只问一题：取 $\beta=0.8$，初始 $m_0=0$，连续两个梯度 $g_1=g_2=5$。请计算 $m_1$、$m_2$、$\hat m_2=m_2/(1-\beta^2)$，并解释修正后的结果为何是 5。先由学生独立解答，再继续后续 Adam 代码。

## EMA 偏差修正理解检查：学生作答与验收

学生原始回答：

> m1 = 1 m2 = 0.8 + 1 = 1.8 因为 /1-beta^2 相当于对 m_2进行统计意义的无偏修正

**评价：** $m_1=1$、$m_2=1.8$ 两步计算正确；知道除以 $1-\beta^2$ 是对零初始化造成的偏差修正；但没有显式计算题目要求的 $\hat m_2$，由导师补全如下。

$$m_1=0.8\cdot0+0.2\cdot5=1,$$

$$m_2=0.8\cdot1+0.2\cdot5=1.8,$$

$$\hat m_2=\frac{1.8}{1-0.8^2}=\frac{1.8}{0.36}=5.$$

进一步说明为什么修正后得到 5：两步对应的未归一化权重分别为 $(1-\beta)\beta=0.16$、$(1-\beta)=0.2$，相加是 $1-\beta^2=0.36$。于是偏差修正后的权重之和等于 1。对于常数梯度 5，加权平均必然为 5。

**统计条件：** 如果所有历史梯度具有同一均值 $\mu$，则 $\mathbb E[m_t]=(1-\beta^t)\mu$，而 $\mathbb E[\hat m_t]=\mu$。非平稳训练过程不能保证相对当前梯度分布的完全无偏性。

本理解检查：前两步数值正确，修正原理基本正确，最后一步的数值由导师补充；仍归入补充理解练习，不计入官方 Exercises/Post Quiz。

### 下一项一题一问：Adam 实际更新

设第 1 步梯度 $g_1=10$，Adam 的初始 $m_0=v_0=0$，$\beta_1=0.9$、$\beta_2=0.999$；已知 $m_1=1$、$v_1=0.1$、$\hat m_1=10$、$\hat v_1=100$。令 $\eta=0.001$ 且仅为简化计算取 $\epsilon=0$。请学生计算：

$$\Delta\theta_1=-\eta\frac{\hat m_1}{\sqrt{\hat v_1}+\epsilon}.$$

问：第 1 步参数更新量是多少？为什么它不是普通 GD 对 $g_1=10$ 的 $-0.01$？学生答完再进入多步数值与 Adam 代码实现。

## Adam 第一步参数更新计算（理解检查通过）

学生原始回答：

> delta_theta_one = - 0.001 因为我们对这个进行了幅度修正

**判定：数值正确，概念方向正确；进一步明确偏差修正与梯度尺度归一化的区别。**

已知 $\eta=0.001$、$\hat m_1=10$、$\hat v_1=100$、为本题简化设 $\epsilon=0$：

$$\Delta\theta_1=-\eta\frac{\hat m_1}{\sqrt{\hat v_1}}=-0.001\cdot\frac{10}{10}=-0.001.$$

而普通 GD 对 $g_1=10$ 的更新为 $-0.01$。偏差修正是将 EMA 初始化权重归一化（$m/(1-\beta_1^t)$、$v/(1-\beta_2^t)$）；本题改变更新幅度的直接原因是用 $\sqrt{\hat v_t}+\epsilon$ 对平均梯度进行逐坐标尺度归一化。

特殊首步情形：若 $m_0=v_0=0$，$g_1\neq0$ 且 $\epsilon=0$，偏差修正后有 $\hat m_1=g_1$、$\hat v_1=g_1^2$，因此 $\Delta\theta_1=-\eta\operatorname{sign}(g_1)$。此结论不适用于 $g_1=0$ 或 $\epsilon>0$ 的精确等式，也不代表全部 Adam 迭代。

### 官方 Build It 第 4 步：Adam 状态初始化（等待学生实现）

根据官方源码的参数表，构造器接口为：

```python
class Adam:
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        # 学生自己完成
        pass
```

学生本轮仅需实现构造器，保存 lr、beta1、beta2、epsilon，并初始化一阶/二阶矩状态 m、v 为 None，时间步计数 t 为 0。设计动机：Adam 和 Momentum 一样不知道未来 params 长度，要首次 step 才初始化同维零向量；另外 t 记录已做的更新次数，用于 $1-\beta_1^t$ 与 $1-\beta_2^t$。

请学生同时解释：为什么需要成员变量 t，不能每次调用 step 都重新让 t=1？之后逐段实现 moment EMA 与偏差修正。

In [ ]:
class Adam:
    def __init__(
        self,
        lr=0.001,
        beta1=0.9,
        beta2=0.999,
        epsilon=1e-8
    ):
        # 请你自己完成
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0因为我们在偏差修正的时候需要t来修正

## Adam 构造器：学生首次提交与真实运行检查

学生提交代码保留在上方独立代码单元。此处不将粘贴时附在代码末尾的中文解释偷偷删掉。

**代码结构和数学理解正确，但原始源码尚无法直接运行。** 最后一行 `self.t = 0因为我们在偏差修正的时候需要t来修正` 把中文解释粘在表达式后，实际 Python `compile` 报 `SyntaxError: invalid syntax`（第16行）。学生的意思是把 `t` 初始化为 0，供偏差修正时使用。

为了独立验收算法逻辑，仅将该行解释分离为 `self.t = 0` 后，在 Python 中实际构造类并验证状态：默认参数和自定义参数的两项断言 **2/2 PASS**。其余 6 个属性和初始化值均与官方 `code/optimizers.py` 的 Adam 构造器一致。

**状态判断：** 理解和初始化逻辑合格；原始代码待将中文解释独立成一行或注释，不能标为原始源码可运行。`t` 是累计 `step` 更新次数：构造器中为 0，第一次执行 step 时变 1，第二次变 2；不能在每次调用 step 时重置为 1，否则偏差修正的分母 `1-beta1**t` 和 `1-beta2**t` 错误。

### 下一轮：Adam.step 的初始化片段

先让学生将 `self.t=0` 单独放到合法的代码行。之后实现 `step(self,params,grads)` 的第一段：当 `self.m is None` 时根据 `len(params)` 把 `m`、`v` 建成等长零列表；每次调用时 `self.t += 1`。不要提前代写后面的逐参数一阶矩、二阶矩公式；学生独立完成后再验收。

按 `skills/learn/SKILL.md` 每次讲一个短代码片段、理解确认后再继续。

In [ ]:
# 对学生原始代码的检查（已实际在 Python 运行）
raw_student_last_line = 'self.t = 0因为我们在偏差修正的时候需要t来修正'
# 原始完整代码无法编译，报 SyntaxError（第 16 行）。
# 将解释文字分离后，以下断言用于验收逻辑，不代表原始源码通过。
class Adam:
    def __init__(self,lr=0.001,beta1=0.9,beta2=0.999,epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = None
        self.v = None
        self.t = 0
assert vars(Adam()) == dict(lr=0.001,beta1=0.9,beta2=0.999,epsilon=1e-8,m=None,v=None,t=0)
assert vars(Adam(lr=.02,beta1=.7,beta2=.88,epsilon=1e-6)) == dict(lr=.02,beta1=.7,beta2=.88,epsilon=1e-6,m=None,v=None,t=0)
print('Cleaned constructor: 2/2 PASS; original raw input: SyntaxError')

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.m = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

## Adam 的 step 初始化片段：第一次提交（需要订正）

学生原始代码见上方独立单元。代码判断 `self.m is None and self.v is None`，按参数维度建立零列表，以及每次增加 `self.t` 的方向均正确。
**需要修正**：第二条赋值重复写成 `self.m = [0.0] * len(params)`，导致 `self.v` 仍为 `None`。在本课初始 `m=v=None` 且一起初始化的前提下，联合条件能工作，但官方只判断 `m is None`。

已实际在 Python 中运行原始片段，`params=[1.0,2.0]`、`grads=[0.3,-0.1]`；执行后 `m=[0.0,0.0]`，`v=None`，`t=1`。模拟随后二阶矩的 `list(zip(self.v, grads))`，得到 `TypeError: 'NoneType' object is not iterable`。

**本轮验收：尚未通过，等待订正。** 请只将第二次初始化写入 `self.v`，保留其余代码；不提前展开 Adam 一阶矩与二阶矩更新。

In [ ]:
# 实际检验学生本轮 step 初始化片段（原样保留）
class AdamInitCheck:
    def __init__(self):
        self.m = None
        self.v = None
        self.t = 0
    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.m = [0.0] * len(params)
        self.t += 1
a = AdamInitCheck()
a.step([1.0, 2.0], [0.3, -0.1])
assert a.m == [0.0, 0.0]
assert a.v is None
assert a.t == 1
try:
    list(zip(a.v, [0.3, -0.1]))
except TypeError as exc:
    print('Expected failure:', repr(exc))
else:
    raise AssertionError('Expected TypeError from uninitialized v')

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

## Adam step 初始化订正：通过（6/6 检查）

学生完整原始订正代码见上一代码单元。修复了第一次提交中把第二个赋值误写成 self.m 的问题，现在分别初始化 self.m 和 self.v，并在条件分支外对 self.t 加 1。

**仅此初始化代码片段验收通过，尚未实现 EMA 更新及参数更新。**

实际使用 Python 运行下面的测试（代码保存在下一单元），全部 6/6 通过：

1. 首次调用 m 为零向量；
2. 首次调用 v 为零向量；
3. 首次调用 t=1；
4. 第二次调用保留人为赋值的 m、v 历史状态；
5. 第二次调用 t=2；
6. 单参数 tuple 输入仍创建正确维度的状态。

代码检查：`if self.m is None and self.v is None` 在本课约定下（m、v 总是一同建立）成立；官方写法为 `if self.m is None`，两种写法在正常状态流上等效。`self.t += 1` 放在 if 外，确保每步累加。

In [ ]:
class _AdamStateOnly:
    def __init__(self):
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1

obj = _AdamStateOnly()
obj.step([1., 2., 3.], [.1, .2, .3])
assert obj.m == [0., 0., 0.]
assert obj.v == [0., 0., 0.]
assert obj.t == 1
obj.m = [2., 3., 4.]
obj.v = [5., 6., 7.]
obj.step([1., 2., 3.], [.1, .2, .3])
assert obj.m == [2., 3., 4.] and obj.v == [5., 6., 7.]
assert obj.t == 2
obj2 = _AdamStateOnly()
obj2.step((10.,), (.5,))
assert obj2.m == [0.] and obj2.v == [0.] and obj2.t == 1
print('Adam initialization step: 6/6 PASS')

## Build It 第 4 步：Adam 一阶矩递推（等待学生）

已知 `self.m` 首次调用被初始化为零向量、`self.beta1` 在构造器里保存，当前 `grads` 是各个参数的梯度。

$$m_{t,i}=\beta_1 m_{t-1,i}+(1-\beta_1)g_{t,i}.$$

下一轮只用 `zip(self.m,grads)` 和列表推导式更新 `self.m`，不要提前写二阶矩、偏差修正、参数更新。

先给学生独立尝试机会；检查其代码时注意 `(1-beta1)` 的乘积项不能漏掉。

In [ ]:
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ^ 2 for v,g in zip(self.v, grads)]

## Adam 一阶矩 / 二阶矩递推：学生首次提交与错误分析

**学生原始代码**保存于上一个代码单元。此前只布置一阶矩，学生主动实现了一阶矩和二阶矩。

**判断**：一阶矩递推正确：$m_t=\beta_1m_{t-1}+(1-\beta_1)g_t$。二阶矩的结构正确，但 Python `g ^ 2` 不是乘方，`^` 是按位异或；应使用 `g ** 2`。

实际 Python 运行学生代码：构造器状态正确，调用 `step([1.0,2.0],[2.0,-3.0])` 时抛出 `TypeError("unsupported operand type(s) for ^: 'float' and 'int'")`；发生异常前 `t` 已从0变1，`m` 已更新为约 `[0.2,-0.3]`，`v` 保持 `[0.0,0.0]`，说明该异常可能遗留**部分更新状态**，不能当作一次成功的 step。

在仅把 `g ^ 2` 改成 `g ** 2` 后（只作为导师验收，不替代学生订正），输入两步梯度 `[2,-3]` 和 `[-2,3]`、默认 $\beta_1=0.9$、$\beta_2=0.999$，得到：

- 第一步 $m\approx[0.2,-0.3]$，$v\approx[0.004,0.009]$，$t=1$。
- 第二步 $m\approx[-0.02,0.03]$，$v\approx[0.007996,0.017991]$，$t=2$。

上述结果实际在 Python 中执行，并用浮点容差断言检验成功。代码附在下一单元。**原始提交本轮未通过，等待学生纠正运算符后重新提交**。尚未开始偏差修正或参数更新实现。

In [ ]:
import math

class _AdamMomentRaw:
    def __init__(self):
        self.beta1, self.beta2 = 0.9, 0.999
        self.m = None
        self.v = None
        self.t = 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ^ 2 for v,g in zip(self.v, grads)]

raw = _AdamMomentRaw()
try:
    raw.step([1.0, 2.0], [2.0, -3.0])
except TypeError as exc:
    print('Raw code error:', str(exc))
else:
    raise AssertionError('Expected TypeError due to ^')
assert raw.t == 1 and raw.v == [0.0, 0.0]

class _AdamMomentFixed(_AdamMomentRaw):
    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * (g ** 2) for v,g in zip(self.v, grads)]

fixed = _AdamMomentFixed()
fixed.step([1.0, 2.0], [2.0, -3.0])
assert fixed.t == 1
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.m, [0.2, -0.3]))
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.v, [0.004, 0.009]))
fixed.step([1.0, 2.0], [-2.0, 3.0])
assert fixed.t == 2
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.m, [-0.02, 0.03]))
assert all(math.isclose(a,b,abs_tol=1e-12) for a,b in zip(fixed.v, [0.007996, 0.017991]))
print('Corrected moment update: two steps passed')

### 下一项只修正一个运算符（等待学生）

请学生把二阶矩更新中的 `g ^ 2` 改为 Python 的乘方写法，保持其余代码，重新提交完整递推片段。验收通过后，再按照官方 Build It 第 4 步进入 `m_hat` / `v_hat` 偏差修正与逐参数更新。

In [ ]:
# 2026-11-06：学生原始订正，Adam.step 的状态、一阶矩与二阶矩片段
# 保留学生原始格式，不代表此片段可作为独立顶层代码直接执行。
def step(self, params, grads):
        # 1. 首次调用时初始化 m 和 v
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        # 2. 每次调用时将 t 加 1
        self.t += 1

        self.m = [self.beta1 * m + (1 - self.beta1) * g for m,g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v,g in zip(self.v, grads)]

In [ ]:
# 学生原始偏差修正代码（2026-11-06，正确）
m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]

In [ ]:
# 学生原始参数更新代码（2026-11-06，待订正）
# 这里的列表推导式 for 放错位置，保留作错误记录
new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon for p, m_h ,v_h in zip(params,m_hat,v_hat)) ]
return new_params

## Adam 续学：原始答案、诊断与讲评（2026-11-06）

**递推订正：通过。** 学生已把二阶矩乘方由先前的 `g ^ 2` 改为 `g ** 2`，与公式 $v_t=\beta_2v_{t-1}+(1-\beta_2)g_t^2$ 一致。

**偏差修正：通过。** 学生独立写出：
```python
m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
```

**参数更新：首次提交未通过。** 学生原始代码在前一代码单元保留。表达式虽然可经 Python 语法解析，但 `for ...` 被写在分母括号内，形成生成器表达式，而 `p` 位于生成器作用域之外；独立运行该表达式触发 `NameError: name 'p' is not defined`。导师给出下列参考订正（**非学生提交**）：
```python
new_params = [
    p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon)
    for p, m_h, v_h in zip(params, m_hat, v_hat)
]
return new_params
```

**学生第一次手算回答（原文）：**
> p_new = 2.99 ,因为我们要的v_hat是二阶矩的修正,所以要加上sqrt

数值正确：$p=3,\eta=0.01,\hat m=2,\hat v=4,\epsilon=10^{-8}$，故 $p_{new}=3-0.01\cdot2/(2+10^{-8})\approx2.99000000005$。概念解释不充分：开平方是将梯度平方的尺度恢复为与梯度相同的量纲，从而逐坐标规范化；它不是因为“修正”这一操作本身需要开平方。

**学生第二次理解检查（原文）：**
> 1.情况A的delta_p_a是-0.01 2.情况B的delta_p_b 是-0.001,因为他们的量纲不一样

两个数值均正确。给定 $\hat m_t=0.5$、$\eta=0.01$、忽略 $\epsilon$：A 的 $\hat v_t=0.25$ 导致 $\Delta p_A=-0.01$；B 的 $\hat v_t=25$ 导致 $\Delta p_B=-0.001$。**原因应为二阶矩的数值 / 历史梯度平方尺度不同，而不是量纲不同**：A 与 B 的 $\hat v$ 都有梯度平方的相同量纲。

**实际运行核验（教师独立在 Python 中完成；后附可复验代码单元）：**原始更新表达式运行得 `NameError`；参考订正构成完整 Adam 后，首步两参数输出约为 `[2.99000000005, 4.00999999995]`，第二步约为 `[2.9816940247702366, 4.01773077269116]`，时间步 `t=2`；A/B 手算断言通过。这些是**教师参考实现的测试**，不代表学生原始完整代码已经独立通过。

**课程交付物提醒：** 本课 `outputs/prompt-optimizer-guide.md` 和中文副本已检查。它们作为本课 Ship It 输出和训练任务选择优化器的实践 Prompt，非官方 `skills/learn/SKILL.md` 的替代教学规范。按 `learning-artifacts/cn/AGENTS.md`，续课应先读取 Skill、英文原文、Notebook 和 Quiz，核对完整镜像及输出文件。

In [ ]:
# 教师参考订正 + 独立回归验证（不会覆盖学生原始代码）
import ast
import math

raw_line = "new_params = [p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon for p, m_h ,v_h in zip(params,m_hat,v_hat)) ]"
ast.parse(raw_line)   # 语法可解析，但变量作用域不正确

class _AdamReferenceCheck:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr, self.beta1, self.beta2, self.epsilon = lr, beta1, beta2, epsilon
        self.m, self.v, self.t = None, None, 0

    def step(self, params, grads):
        if self.m is None and self.v is None:
            self.m = [0.0] * len(params)
            self.v = [0.0] * len(params)
        self.t += 1
        self.m = [self.beta1 * m + (1 - self.beta1) * g for m, g in zip(self.m, grads)]
        self.v = [self.beta2 * v + (1 - self.beta2) * g ** 2 for v, g in zip(self.v, grads)]
        m_hat = [m / (1 - self.beta1 ** self.t) for m in self.m]
        v_hat = [v / (1 - self.beta2 ** self.t) for v in self.v]
        return [
            p - self.lr * m_h / (math.sqrt(v_h) + self.epsilon)
            for p, m_h, v_h in zip(params, m_hat, v_hat)
        ]

ref = _AdamReferenceCheck()
first = ref.step([3.0, 4.0], [2.0, -2.0])
second = ref.step(first, [0.5, -0.3])
assert all(math.isclose(a, b, abs_tol=1e-10) for a, b in zip(first, [2.99, 4.01]))
assert ref.t == 2
assert math.isclose(-0.01 * 0.5 / math.sqrt(0.25), -0.01)
assert math.isclose(-0.01 * 0.5 / math.sqrt(25), -0.001)
print("Teacher reference check passed:", first, second, "t =", ref.t)

### 当前进度与下一步（未完成）

Adam 的 EMA 和偏差修正已经由学生独立提交并通过。参数更新的列表推导式有首次原始提交和教师参考纠错，**待学生确认其订正后的完整 `step()` 并运行真实测试**。随后做 Rosenbrock 优化器对比、与 PyTorch 核验、官方 Exercises 和 Post Quiz。整课仍为**进行中**，根目录 `LEARNING.md` 暂不增加“已完成”记录。